# 02. Model implementation, selection and tuning
Start with notebook 01. Use the same Drive project folder. Run one stage cell at a time and share its output for review. Never use Run all across stages.


## Connect Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Pull repository updates
Stop if Git reports an error. Resolve conflicts before continuing; no files are reset.


In [ ]:
%%bash
set -e
cd /content/drive/MyDrive/FoodPriceWatch
git pull --no-rebase origin main


## Initialize project runtime
Persistent artifacts are saved under the Drive project folder.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, hashlib, csv
ROOT = Path('/content/drive/MyDrive/FoodPriceWatch')
assert (ROOT / 'src/fpw/runner.py').is_file(), 'Repository is missing on Drive'
os.chdir(ROOT)
os.environ['PYTHONPATH'] = str(ROOT / 'src')
sys.path.insert(0, str(ROOT / 'src'))
def git(*args):
    return subprocess.check_output(['git', *args], cwd=ROOT, text=True).strip()


## 3. Install runtime packages


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(ROOT / 'requirements-colab.txt')], check=True)
os.environ.setdefault('FPW_THREADS', str(os.cpu_count() or 2))
print('Package install: PASS; threads:', os.environ['FPW_THREADS'])


## 4. Startup verification
A failed check stops execution. Missing or changed finished outputs are reset to pending by the project checker; paste failures back before continuing.


In [ ]:
for folder in ['outputs', 'data']:
    for tmp in (ROOT / folder).rglob('*.tmp'):
        assert tmp.resolve().is_relative_to(ROOT.resolve())
        tmp.unlink()
        print('Removed interrupted-write temporary file:', tmp.relative_to(ROOT))
subprocess.run([sys.executable, '-m', 'fpw.check'], cwd=ROOT, check=True)
def run_unit(sid):
    subprocess.run([sys.executable, '-m', 'fpw.runner', sid], cwd=ROOT, check=True)
    ledger = json.loads((ROOT / 'state/stages.json').read_text())
    stage = next(s for s in ledger['stages'] if s['id'] == sid)
    assert stage['status'] == 'done', 'STOP: stage was not finalized'
    summary = []
    for name, expected in stage['output_sha256'].items():
        path = ROOT / name
        actual = hashlib.sha256(path.read_bytes()).hexdigest()
        assert actual == expected, 'STOP: output hash mismatch: ' + name
        rows = None
        if path.suffix == '.csv':
            with path.open(newline='') as f:
                rows = sum(1 for _ in csv.DictReader(f))
        elif path.suffix == '.parquet':
            import pyarrow.parquet as pq
            rows = pq.read_metadata(path).num_rows
        summary.append(dict(file=name, rows=rows, sha256=actual))
    assert summary, 'STOP: no finalized output hashes'
    print(json.dumps(dict(unit=sid, outputs=summary, check='PASS', git_commit=git('rev-parse', 'HEAD')), indent=2))


## S24: Stacked-model implementation checks
**Not implemented yet.** No execution command is provided. Prerequisites and required decisions must be verified before this stage is implemented.


## S25: Validation-based model selection
**Not implemented yet.** No execution command is provided. Prerequisites and required decisions must be verified before this stage is implemented.


## S26: Tuning and validation confirmation
**Not implemented yet.** No execution command is provided. Prerequisites and required decisions must be verified before this stage is implemented.


Testing requires committed pre-registration and existing checkpoint enforcement. Do not rerun earlier sealed tests. Keep models, exports and data out of GitHub.
